In [ ]:
from segment_anything import SamPredictor, sam_model_registry
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import numpy as np
import imageio

In [ ]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))

In [ ]:
sam = sam_model_registry["default"](checkpoint="sam_vit_h_4b8939.pth")
device = "cuda"
sam.to(device=device)
predictor = SamPredictor(sam)

In [ ]:
n = '02'

In [ ]:
image_path = f'selected_calib_photos/{n}.JPG'
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
predictor.set_image(image)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
plt.axis('on')
plt.show()

In [ ]:
input_point = np.array([[1000, 700]])
input_label = np.array([1])
plt.figure(figsize=(10,10))
plt.imshow(image)
show_points(input_point, input_label, plt.gca())
plt.axis('on')
plt.show()

In [ ]:
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=False,
)

In [ ]:
for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(10,10))
    plt.imshow(image)
    mask_colored = np.where(mask == 1, 255, mask)
    mask_image = show_mask(mask_colored, plt.gca())
    show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
    plt.axis('off')
    plt.show()

In [ ]:
color_mask = np.zeros_like(image)
color_mask[mask > 0.5] = [255, 255, 255]
masked_image = cv2.addWeighted(image, 1, color_mask, 1, 0)

cv2.imwrite(f'_masked_{n}.png', cv2.cvtColor(masked_image, cv2.COLOR_RGB2BGR))

In [ ]:
mask_bool = color_mask.astype(bool)
black_background = np.zeros_like(image)
black_background[mask_bool] = masked_image[mask_bool]

In [ ]:
image_rgb = cv2.cvtColor(black_background, cv2.COLOR_BGR2RGB)
image_pil = Image.fromarray(image_rgb)
display(image_pil)

In [ ]:
cv2.imwrite(f'masked_{n}.png', cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))